# Source Classification

This notebook trains and tests the SmogNet probable-source classifier for detected pollution spikes.

Important note about accuracy: the supplied data does not include confirmed labels such as `Crop Burning`, `Vehicular Emissions`, or `Dust Storm`. The notebook therefore reports **proxy accuracy** against transparent pollutant-fingerprint labels. This is a validation check for consistency and reasonableness, not a claim of confirmed source attribution.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.anomaly_detection import (
    compute_rolling_baselines,
    detect_spikes,
    learn_anomaly_thresholds,
    score_test_data,
)
from src.config import DEFAULT_ROLLING_WINDOW, RAW_DATA_DIR
from src.data_loader import load_air_quality_data
from src.preprocessing import detect_pollutant_columns, preprocess_air_quality_data

from src.source_classification import (
    attach_dust_ratio_thresholds,
    classify_all_spikes,
    fit_dust_ratio_reference,
)

pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PROJECT_ROOT

PosixPath('/Users/waleedhassan/Downloads/Datathon/Somgnet-V1.1.1')

## 1. Load Data and Detect Test Spikes

Source classification runs after spike detection, so this notebook first rebuilds the same trained spike detector and applies it to the test split.

In [2]:
raw_train, raw_test, split_metadata = load_air_quality_data(RAW_DATA_DIR)

train_df = preprocess_air_quality_data(raw_train, is_train=True)
test_df = preprocess_air_quality_data(raw_test, is_train=False)

pollutant_cols = [
    pollutant
    for pollutant in detect_pollutant_columns(train_df)
    if pollutant in test_df.columns
]

train_scored, baseline_reference = compute_rolling_baselines(
    train_df,
    pollutant_cols,
    window=DEFAULT_ROLLING_WINDOW,
)
thresholds = learn_anomaly_thresholds(train_scored, pollutant_cols)
test_scored = score_test_data(test_df, baseline_reference, pollutant_cols)
detected_spikes = detect_spikes(test_scored, thresholds, pollutant_cols)

print(f"Training rows: {len(train_df):,}")
print(f"Testing rows: {len(test_df):,}")
print(f"Detected spikes available for source classification: {len(detected_spikes):,}")
print(f"Pollutants used: {pollutant_cols}")

detected_spikes.head()

[load] Warning: trimming 18305 training row(s) at or after the test start 2024-07-01 00:00:00 to prevent temporal leakage.
Training rows: 122,896
Testing rows: 21,792
Detected spikes available for source classification: 3,763
Pollutants used: ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']


,timestamp,city,season,co,no,no2,o3,so2,pm2_5,pm10,nh3,co_score,no_score,no2_score,o3_score,so2_score,pm2_5_score,pm10_score,nh3_score,combined_anomaly_score,dominant_pollutant,severity,anomaly_explanation,datetime,main_aqi,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,shortwave_radiation,hour,day,month,year,date,co_baseline_median,co_baseline_iqr,co_baseline_level,no_baseline_median,no_baseline_iqr,no_baseline_level,no2_baseline_median,no2_baseline_iqr,no2_baseline_level,o3_baseline_median,o3_baseline_iqr,o3_baseline_level,so2_baseline_median,so2_baseline_iqr,so2_baseline_level,pm2_5_baseline_median,pm2_5_baseline_iqr,pm2_5_baseline_level,pm10_baseline_median,pm10_baseline_iqr,pm10_baseline_level,nh3_baseline_median,nh3_baseline_iqr,nh3_baseline_level,co_moderate_threshold,co_high_threshold,co_severe_threshold,co_threshold_level,co_is_anomalous,no_moderate_threshold,no_high_threshold,no_severe_threshold,no_threshold_level,no_is_anomalous,no2_moderate_threshold,no2_high_threshold,no2_severe_threshold,no2_threshold_level,no2_is_anomalous,o3_moderate_threshold,o3_high_threshold,o3_severe_threshold,o3_threshold_level,o3_is_anomalous,so2_moderate_threshold,so2_high_threshold,so2_severe_threshold,so2_threshold_level,so2_is_anomalous,pm2_5_moderate_threshold,pm2_5_high_threshold,pm2_5_severe_threshold,pm2_5_threshold_level,pm2_5_is_anomalous,pm10_moderate_threshold,pm10_high_threshold,pm10_severe_threshold,pm10_threshold_level,pm10_is_anomalous,nh3_moderate_threshold,nh3_high_threshold,nh3_severe_threshold,nh3_threshold_level,nh3_is_anomalous,is_anomaly
0,2024-07-01 00:00:00,Karachi,Monsoon/Summer,257.020,0.000,1.540,46.490,1.560,42.220,75.170,0.590,-0.159,-0.069,-0.702,0.901,-0.444,3.476,0.016,0.068,3.476,pm2_5,high,PM2_5 is 3.48 robust-IQR units above its city ...,1/7/2024 0:00,3,30.306,87.804,28.056,0.000,994.016,13.708,256.329,0,0,1,7,2024,2024-07-01,268.700,73.438,city_season,0.090,1.295,city_season,4.520,4.245,city_season,37.550,9.928,city_season,1.940,0.855,city_season,20.875,6.140,city_season,74.750,26.150,city_season,0.555,0.515,city_season,3.523,6.994,13.687,city_season,False,4.243,8.962,20.291,city_season,False,2.778,4.474,7.780,city_season,False,1.606,2.226,3.087,city_season,False,3.351,5.344,9.315,city_season,False,2.450,3.459,6.059,city_season,True,2.362,3.085,5.065,city_season,False,3.498,6.300,11.927,city_season,False,True
1,2024-07-01 01:00:00,Karachi,Monsoon/Summer,250.340,0.000,1.690,45.420,1.670,47.800,90.920,0.460,-0.250,-0.069,-0.667,0.793,-0.316,4.385,0.618,-0.184,4.385,pm2_5,high,PM2_5 is 4.39 robust-IQR units above its city ...,1/7/2024 1:00,3,30.607,87.320,28.256,0.000,993.817,16.236,273.814,1,1,1,7,2024,2024-07-01,268.700,73.438,city_season,0.090,1.295,city_season,4.520,4.245,city_season,37.550,9.928,city_season,1.940,0.855,city_season,20.875,6.140,city_season,74.750,26.150,city_season,0.555,0.515,city_season,3.523,6.994,13.687,city_season,False,4.243,8.962,20.291,city_season,False,2.778,4.474,7.780,city_season,False,1.606,2.226,3.087,city_season,False,3.351,5.344,9.315,city_season,False,2.450,3.459,6.059,city_season,True,2.362,3.085,5.065,city_season,False,3.498,6.300,11.927,city_season,False,True
2,2024-07-01 02:00:00,Karachi,Monsoon/Summer,253.680,0.000,1.840,46.490,1.460,50.060,104.860,0.370,-0.205,-0.069,-0.631,0.901,-0.561,4.753,1.151,-0.359,4.753,pm2_5,high,PM2_5 is 4.75 robust-IQR units above its city ...,1/7/2024 2:00,4,30.907,84.842,28.056,0.000,993.618,18.014,267.709,60,2,1,7,2024,2024-07-01,268.700,73.438,city_season,0.090,1.295,city_season,4.520,4.245,city_season,37.550,9.928,city_season,1.940,0.855,city_season,20.875,6.140,city_season,74.750,26.150,city_season,0.555,0.515,city_season,3.523,6.994,13.687,city_season,False,4.243,8.962,20.291,city_season,False,2.778,4.474,7.780,city_season,False,1.606,2.226,3.087,city_season,False,3.351,5.344,9.315,city_season,False,2.450,3.459,6.059,city_season,True,2.362,3.085,5.065,city_season

## 2. Train and Apply the Source Classifier

The source classifier is intentionally interpretable. The trainable part is the PM10/PM2.5 dust-ratio reference, which is learned from training data only. The remaining source fingerprints are rule-based and use normalized anomaly evidence.

In [3]:
dust_reference = fit_dust_ratio_reference(train_df)
spikes_with_ratios = attach_dust_ratio_thresholds(detected_spikes, dust_reference)
classified_spikes = classify_all_spikes(spikes_with_ratios)

classified_spikes[[
    "timestamp",
    "city",
    "severity",
    "dominant_pollutant",
    "probable_source",
    "source_confidence",
    "source_reason",
    "crop_score",
    "vehicular_score",
    "industrial_score",
    "dust_score",
]].head(10)

,timestamp,city,severity,dominant_pollutant,probable_source,source_confidence,source_reason,crop_score,vehicular_score,industrial_score,dust_score
0,2024-07-01 00:00:00,Karachi,high,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
1,2024-07-01 01:00:00,Karachi,high,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
2,2024-07-01 02:00:00,Karachi,high,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
3,2024-07-01 03:00:00,Karachi,high,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
4,2024-07-01 04:00:00,Karachi,moderate,pm2_5,Vehicular Emissions,Low,Probable traffic fingerprint: NO and NO2 both ...,0.000,0.047,0.000,0.000
5,2024-07-01 06:00:00,Quetta,severe,o3,Industrial Emissions,Low,Probable industrial fingerprint: SO2 showed th...,0.120,0.013,0.347,0.000
6,2024-07-01 07:00:00,Quetta,moderate,o3,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
7,2024-07-01 08:00:00,Karachi,moderate,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
8,2024-07-01 09:00:00,Karachi,high,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000
9,2024-07-01 10:00:00,Karachi,high,pm2_5,Mixed Sources,Low,Overlapping pollutant fingerprints were presen...,0.000,0.000,0.000,0.000


## 3. Check Proxy Accuracy

The proxy label below is generated from strict pollutant fingerprints:

- Crop Burning: NH3 and CO are both anomalous.
- Vehicular Emissions: NO and NO2 are both anomalous.
- Industrial Emissions: SO2 is anomalous and is the dominant pollutant.
- Dust Storm: PM10 is anomalous, PM10/PM2.5 exceeds the learned dust-ratio threshold, and PM10 evidence is stronger than PM2.5.
- Mixed Sources: no single strict fingerprint dominates or multiple fingerprints overlap.

In [4]:
def source_proxy_label(row: pd.Series) -> str:
    fingerprints = []

    if bool(row.get("nh3_is_anomalous", False)) and bool(row.get("co_is_anomalous", False)):
        fingerprints.append("Crop Burning")

    if bool(row.get("no_is_anomalous", False)) and bool(row.get("no2_is_anomalous", False)):
        fingerprints.append("Vehicular Emissions")

    if bool(row.get("so2_is_anomalous", False)) and row.get("dominant_pollutant") == "so2":
        fingerprints.append("Industrial Emissions")

    ratio = row.get("pm10_pm2_5_ratio", np.nan)
    ratio_threshold = row.get("dust_ratio_threshold", np.nan)
    if (
        bool(row.get("pm10_is_anomalous", False))
        and pd.notna(ratio)
        and pd.notna(ratio_threshold)
        and ratio > ratio_threshold
        and row.get("pm10_score", 0) > row.get("pm2_5_score", 0)
    ):
        fingerprints.append("Dust Storm")

    unique_fingerprints = sorted(set(fingerprints))
    if len(unique_fingerprints) == 1:
        return unique_fingerprints[0]
    return "Mixed Sources"


classified_spikes = classified_spikes.copy()
classified_spikes["proxy_source_label"] = classified_spikes.apply(source_proxy_label, axis=1)

source_labels = [
    "Crop Burning",
    "Vehicular Emissions",
    "Industrial Emissions",
    "Dust Storm",
    "Mixed Sources",
]

source_accuracy = accuracy_score(
    classified_spikes["proxy_source_label"],
    classified_spikes["probable_source"],
)

summary_metrics = pd.DataFrame(
    [
        {
            "proxy_source_accuracy": source_accuracy,
            "classified_spikes": len(classified_spikes),
            "non_mixed_rate": (classified_spikes["probable_source"] != "Mixed Sources").mean(),
            "medium_or_high_confidence_rate": classified_spikes["source_confidence"].isin(["Medium", "High"]).mean(),
        }
    ]
)

summary_metrics

,proxy_source_accuracy,classified_spikes,non_mixed_rate,medium_or_high_confidence_rate
0,0.490,3763,0.667,0.256


In [5]:
cm = confusion_matrix(
    classified_spikes["proxy_source_label"],
    classified_spikes["probable_source"],
    labels=source_labels,
)
cm_df = pd.DataFrame(cm, index=source_labels, columns=source_labels)

display(cm_df)

report = classification_report(
    classified_spikes["proxy_source_label"],
    classified_spikes["probable_source"],
    labels=source_labels,
    zero_division=0,
    output_dict=True,
)
pd.DataFrame(report).T

,Crop Burning,Vehicular Emissions,Industrial Emissions,Dust Storm,Mixed Sources
Crop Burning,37,0,0,0,113
Vehicular Emissions,0,22,0,0,7
Industrial Emissions,0,0,593,0,13
Dust Storm,0,0,1,71,1
Mixed Sources,402,225,1148,10,1120


,precision,recall,f1-score,support
Crop Burning,0.084,0.247,0.126,150.000
Vehicular Emissions,0.089,0.759,0.159,29.000
Industrial Emissions,0.340,0.979,0.505,606.000
Dust Storm,0.877,0.973,0.922,73.000
Mixed Sources,0.893,0.386,0.539,"2,905.000"
accuracy,0.490,0.490,0.490,0.490
macro avg,0.457,0.668,0.450,"3,763.000"
weighted avg,0.765,0.490,0.521,"3,763.000"


## 4. Inspect Classification Behavior

These tables show whether the model is producing usable source categories or mostly falling back to mixed/low-confidence classifications.

In [6]:
source_distribution = classified_spikes["probable_source"].value_counts().rename_axis("probable_source").to_frame("count")
confidence_distribution = classified_spikes["source_confidence"].value_counts().rename_axis("source_confidence").to_frame("count")
city_source_summary = (
    classified_spikes.groupby(["city", "probable_source"])
    .size()
    .unstack(fill_value=0)
)

display(source_distribution)
display(confidence_distribution)
display(city_source_summary)

,count
probable_source,
Industrial Emissions,1742
Mixed Sources,1254
Crop Burning,439
Vehicular Emissions,247
Dust Storm,81


,count
source_confidence,
Low,2799
Medium,775
High,189


probable_source,Crop Burning,Dust Storm,Industrial Emissions,Mixed Sources,Vehicular Emissions
city,,,,,
Islamabad,134,0,214,128,22
Karachi,73,33,400,568,101
Lahore,195,5,285,172,96
Peshawar,1,0,475,33,11
Quetta,36,43,368,353,17


In [7]:
fig = px.bar(
    classified_spikes.groupby(["city", "probable_source"]).size().reset_index(name="count"),
    x="city",
    y="count",
    color="probable_source",
    barmode="stack",
    title="Probable Source Counts by City",
)
fig.update_layout(template="plotly_white")
fig.show()

In [8]:
fig = px.scatter(
    classified_spikes,
    x="timestamp",
    y="city",
    color="probable_source",
    symbol="source_confidence",
    size="combined_anomaly_score",
    hover_data=["dominant_pollutant", "severity", "source_reason"],
    title="Source-Classified Spikes Across the Test Period",
)
fig.update_layout(template="plotly_white")
fig.show()